In [2]:
from fractal_printer.mesh import fractal_sdfs as fs, mesh_generation as mg
import numpy as np
from importlib import reload
import quaternion
import sdf
import matplotlib.pyplot as plt
from fractal_printer.paths import OUTPUT_DIR

In [16]:
# Platonic

settings = {
    "coefficients": [
        [0.193, -0.178, -0.513, 0.858],
        [0.387, -0.094, -0.741, 0.148],
        [0.445, 0.004, -0.761, 0.887],
        [-0.464, -0.378, -0.908, -0.427],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0]
    ],
    "power": 3,
    "slice": 0.13400000000000012,
    "offset": 0.002,
    "iterations": 18
}

julia = fs.polynomial_julia_sdf(**settings).rotate(np.pi/2,sdf.X).translate((0,0.2,0))
d12 = sdf.dodecahedron(r=0.7).rotate((180-108)/2 * np.pi/180, sdf.X).translate((0,-0.5,0)).rotate(np.pi/6, sdf.Y)

mg.generate_mesh(
    sdf = sdf.union(julia, d12, k = 0.25),
    samples = 2**30,
    recursion_levels = 20,
    tol = 1e-5,
    simplify = 0.95,
    bounds = mg.box_bounds(size=3),
    save_path=OUTPUT_DIR / "platonic.stl"
)



min -1.5, -1.5, -1.5
max 1.5, 1.5, 1.5
step 0.00292969, 0.00292969, 0.00292969
1174241375 samples in 32768 batches with 1 workers
  100% (32768 of 32768) [##############################] 0:01:40 0:00:00    
25892 skipped, 4456 empty, 2420 nonempty
5241422 triangles in 99.9081 seconds
Bisecting 2620693 unique edge crossings (deduped from 15724266, up to 20 levels, tol=1e-05)...
  3 of 2620693 edges skipped (degenerate/non-finite), kept linear interpolation
	Level 0: 2620690 active edges
	Level 1: 2418967 active edges
	Level 2: 2067967 active edges
	Level 3: 1527962 active edges
	Level 4: 945513 active edges
	Level 5: 520058 active edges
	Level 6: 169313 active edges
	Level 7: 5307 active edges
	Level 8: 281 active edges
	Level 9: 249 active edges
	Level 10: 249 active edges
Converting mesh...
Simplifying mesh by 0.95x ...
Saving mesh to C:\Users\owenb\Code\personal\fractal-printer\outputs\platonic.stl...


<meshio mesh object>
  Number of points: 130994
  Number of cells:
    triangle: 262070

In [69]:
# Vapor Pot

settings = {
    "coefficients": [
        [0.806, 0.587, -0.575, 0.059],
        [0.477, -0.681, 0.102, 0.66],
        [-0.533, 0.637, -0.342, -0.805],
        [0.974, 0.208, -0.735, 0.17]
    ],
    "power": 3,
    "slice": 0.134,
    "offset": 0.002,
    "iterations": 18
}

julia = fs.polynomial_julia_sdf(**settings).translate(sdf.X*-0.5).translate(-sdf.Z*0.05)

# Define the pot
pot_form = sdf.ellipsoid((1.2,0.9,0.9)).translate(sdf.X*0.1) & sdf.slab(x1=1)
pot_inner = pot_form.erode(0.03)
holes = sdf.capped_cylinder(sdf.Z*1.2,sdf.Z*0.9,radius=0.05).circular_array(6,0.2).rotate(np.pi/2,sdf.Y)
pot = ((pot_form - pot_inner) & sdf.slab(x0=0.1)) - holes

# Cutout for some extra bits of the fractal that stick into the pot space
clear_region = (pot_inner & sdf.slab(x0=-0.12)) | sdf.capped_cylinder(a=0, b=sdf.X*-0.2,radius=0.4).translate(-0.3*(sdf.Y+sdf.Z))

# Define the saucer
pegs = sdf.capped_cylinder(sdf.Z*1.06,sdf.Z*1,radius=0.05).circular_array(6,0.4).rotate(np.pi/2,sdf.Y)
dish = (pot_form.dilate(0.06) - pot_form.dilate(0.03)) & sdf.slab(x0=0.95) | pegs

mg.generate_mesh(
    sdf = sdf.union(julia, pot, k = 0.2) - clear_region,
    samples = 2**30,
    recursion_levels = 20,
    tol = 1e-5,
    simplify = 0.95,
    bounds = mg.box_bounds(size=3),
    save_path=OUTPUT_DIR / "vapor_pot.stl"
)
mg.generate_mesh(
    sdf = dish,
    samples = 2**30,
    recursion_levels = 20,
    tol = 1e-5,
    simplify = 0.95,
    bounds = mg.box_bounds(size=3),
    save_path=OUTPUT_DIR / "vapor_pot_dish.stl"
)

min -1.5, -1.5, -1.5
max 1.5, 1.5, 1.5
step 0.00292969, 0.00292969, 0.00292969
1174241375 samples in 32768 batches with 1 workers
  100% (32768 of 32768) [##############################] 0:05:07 0:00:00    
26857 skipped, 3648 empty, 2263 nonempty
6519712 triangles in 306.659 seconds
Bisecting 3259909 unique edge crossings (deduped from 19559136, up to 20 levels, tol=1e-05)...
  8 of 3259909 edges skipped (degenerate/non-finite), kept linear interpolation
	Level 0: 3259901 active edges
	Level 1: 3153017 active edges
	Level 2: 2956402 active edges
	Level 3: 2614484 active edges
	Level 4: 2084982 active edges
	Level 5: 1317293 active edges
	Level 6: 665410 active edges
	Level 7: 30519 active edges
	Level 8: 246 active edges
	Level 9: 38 active edges
	Level 10: 38 active edges
Converting mesh...
Simplifying mesh by 0.95x ...
Saving mesh to C:\Users\owenb\Code\personal\fractal-printer\outputs\vapor_pot.stl...


c:\Users\owenb\Code\personal\fractal-printer\.venv\Lib\site-packages\meshio\stl\_stl.py:206: RuntimeWarning: invalid value encountered in divide
  normals = (normals.T / nrm).T


min -1.5, -1.5, -1.5
max 1.5, 1.5, 1.5
step 0.00292969, 0.00292969, 0.00292969
1174241375 samples in 32768 batches with 1 workers
  100% (32768 of 32768) [##############################] 0:00:46 0:00:00    
32256 skipped, 132 empty, 380 nonempty
979872 triangles in 46.149 seconds
Bisecting 489938 unique edge crossings (deduped from 2939616, up to 20 levels, tol=1e-05)...
	Level 0: 489938 active edges
	Level 1: 487666 active edges
	Level 2: 482522 active edges
	Level 3: 472310 active edges
	Level 4: 311701 active edges
	Level 5: 275541 active edges
	Level 6: 194605 active edges
	Level 7: 1352 active edges
Converting mesh...
Simplifying mesh by 0.95x ...
Saving mesh to C:\Users\owenb\Code\personal\fractal-printer\outputs\vapor_pot_dish.stl...


<meshio mesh object>
  Number of points: 24498
  Number of cells:
    triangle: 48992